# Homothetic Tube MPC — Interactive Lane-Change Playground

**Lateral vehicle dynamics · HT-MPC (proposed, homothetic tube) vs RT-MPC (fixed-tube robust MPC)**

Design your own lane-change manoeuvre and **watch both controllers drive it, step by step**:
the vehicle, the tube cross-section it is currently guaranteed to stay in, and the
prediction the MPC is making at that instant — the nominal trajectory $z_{j|k}$ with the
tube $\alpha_{j|k}\Omega$ around it.

Both controllers share the *same* disturbance bound and the *same* RPI ellipsoid $\Omega$.
The only difference: HT-MPC scales its tube by $\alpha_k = \|e_k\|_{P^{-1}}$ at every step,
RT-MPC always uses the full worst-case tube ($\alpha_k \equiv 1$).

> **Reproducing the paper figures?** Use `LatVeh_Benchmark.ipynb` and `CSTR_Benchmark.ipynb`
> — those are the frozen reference implementations. This notebook imports `htmpc_latveh.py`,
> whose first section is a verbatim copy of the benchmark's model, uncertainty description,
> RPI computation, tightening and OCPs; only the reference generator, the prediction
> recording and the plots are new.

**How to use**

1. Run all cells (`Runtime ▸ Run all` in Colab, `Kernel ▸ Restart & Run All` in Jupyter).
2. In **Interactive playground**, pick a manoeuvre, move the sliders, press **Run**
   (~1 s), then press **▶ Play** on the figure or drag the time slider.

**What to look for**

* Top panel (HT-MPC): the thick bar at the vehicle is the *current* tube cross-section
  $\alpha_k\Omega$ — usually tiny. The shaded band ahead is the tube the OCP is planning with.
* Middle panel (RT-MPC): the current cross-section is always the full $\Omega$, and every
  applied input is tightened by the full worst case — that is why it steers much less
  aggressively (compare the RMS tracking error in the summary).
* Make the manoeuvre aggressive (slalom, short `t_hold`) and RT-MPC loses feasibility
  while HT-MPC keeps going; its panel freezes and the title says at which step.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Installs missing packages (needed on Colab) and fetches the engine module if this
# notebook is not sitting next to htmpc_latveh.py (e.g. opened directly in Colab).
import importlib.util, subprocess, sys, os, urllib.request

RAW_URL = "https://raw.githubusercontent.com/MiladBnit/homothetic-tube-mpc/main/htmpc_latveh.py"

_missing = [p for p in ("cvxpy", "clarabel", "osqp", "scs", "plotly", "ipywidgets", "nbformat")
            if importlib.util.find_spec(p) is None]
if _missing:
    print("Installing:", " ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

if not os.path.exists("htmpc_latveh.py"):
    print("Downloading htmpc_latveh.py ...")
    urllib.request.urlretrieve(RAW_URL, "htmpc_latveh.py")

import numpy as np
import htmpc_latveh as H
print("Engine OK")

## 1 · Offline design (identical numbers to the paper)

Vehicle at $v_{x0} = 60$ km/h, $T_s = 0.1$ s, ancillary LQR gain $K$, shared 99 %
confidence bound, RPI ellipsoid $\Omega = \{e : e^\top P^{\star -1} e \le 1\}$, constraint
tightening and both OCPs with horizon $N = 25$. Runs once (~2 s).

In [ ]:
d = H.design(verbose=True)

## 2 · Interactive playground

| Control | Meaning | Used by |
|---|---|---|
| **Manoeuvre** | preset or custom waypoint list | all |
| **t_lc** | time of the first lane change [s] | single, double, slalom |
| **t_hold** | time in the new lane before switching back [s] | double, slalom |
| **y_lane** | half lane offset: lanes at $\pm$ `y_lane` [m] (paper: 1.75) | single, double, slalom |
| **ramp** | linear ramp replacing each step [s]; 0 = step (paper) | all |
| **T_sim** | simulation length [s] | all |
| **custom** | `t:y; t:y; ...` (seconds : metres), e.g. `0:-1.75; 1.2:1.75; 3.5:0` | custom |
| **plant / noise seed** | realisation of $\Delta v_x$ and of the additive noise $w_k$ | all |
| **speed** | playback speed (1× = real time) | — |

Press **Run**, then use **▶ Play** / the time slider on the figure.

In [ ]:
import ipywidgets as W
from IPython.display import display

_st, _ly = {"description_width": "90px"}, W.Layout(width="420px")
w_mode  = W.Dropdown(options=H.MANOEUVRES, value=H.MANOEUVRES[0], description="Manoeuvre",
                     style=_st, layout=_ly)
w_tlc   = W.FloatSlider(1.2, min=0.2, max=4.0, step=0.1, description="t_lc [s]",
                        style=_st, layout=_ly, readout_format=".1f")
w_hold  = W.FloatSlider(2.0, min=0.3, max=5.0, step=0.1, description="t_hold [s]",
                        style=_st, layout=_ly, readout_format=".1f")
w_ylane = W.FloatSlider(1.75, min=0.25, max=float(np.floor(d.y_ref_max*20)/20), step=0.05,
                        description="y_lane [m]", style=_st, layout=_ly, readout_format=".2f")
w_ramp  = W.FloatSlider(0.0, min=0.0, max=2.0, step=0.1, description="ramp [s]",
                        style=_st, layout=_ly, readout_format=".1f")
w_tsim  = W.FloatSlider(5.0, min=2.0, max=15.0, step=0.5, description="T_sim [s]",
                        style=_st, layout=_ly, readout_format=".1f")
w_cust  = W.Text("0:-1.75; 1.2:1.75; 3.5:0.0", description="custom", style=_st, layout=_ly)
w_pseed = W.IntText(42, description="plant seed", style=_st, layout=W.Layout(width="190px"))
w_nseed = W.IntText(7,  description="noise seed", style=_st, layout=W.Layout(width="190px"))
w_speed = W.Dropdown(options=[("0.5x", 0.5), ("1x", 1.0), ("2x", 2.0), ("4x", 4.0)], value=1.0,
                     description="speed", style=_st, layout=W.Layout(width="190px"))
w_stat  = W.Checkbox(False, description="also show static summary plot", indent=False)
w_run   = W.Button(description="Run", button_style="success", icon="play",
                   layout=W.Layout(width="120px"))
out     = W.Output()


def _on_run(_):
    with out:
        out.clear_output(wait=True)
        try:
            tr_p, tr_r, y_ref, dv = H.run(d, w_mode.value, w_tlc.value, w_hold.value,
                                          w_ylane.value, w_tsim.value, w_ramp.value,
                                          w_cust.value, w_pseed.value, w_nseed.value)
        except ValueError as e:
            print(f"Invalid manoeuvre: {e}")
            return
        H.print_summary(tr_p, tr_r, dv)
        ttl = f"{w_mode.value}, ramp {w_ramp.value:.1f} s"
        H.animate(d, tr_p, tr_r, y_ref, title=ttl,
                  frame_ms=int(round(1000*d.Ts/w_speed.value))).show()
        if w_stat.value:
            H.plot_static(d, tr_p, tr_r, y_ref, title=ttl)


w_run.on_click(_on_run)
display(W.VBox([w_mode, w_tlc, w_hold, w_ylane, w_ramp, w_tsim, w_cust,
                W.HBox([w_pseed, w_nseed, w_speed]), W.HBox([w_run, w_stat]), out]))
_on_run(None)   # paper case on first run

## 3 · Scripted use (no widgets)

The engine can be driven directly — for batch experiments, or if widgets do not render in
your viewer. `H.run` returns the two closed-loop traces (including the per-step predictions
`z_pred`, `a_pred`), the reference and the drawn $\Delta v_x$.

In [ ]:
# Aggressive slalom: RT-MPC becomes infeasible, HT-MPC completes the manoeuvre
tr_p, tr_r, y_ref, dv = H.run(d, "Slalom", t_lc=1.0, t_hold=1.2, T_sim=8.0)
H.print_summary(tr_p, tr_r, dv)
H.animate(d, tr_p, tr_r, y_ref, title="slalom, t_hold = 1.2 s").show()

In [ ]:
# Custom manoeuvre with smooth (ramped) transitions - static summary figure
tr_p, tr_r, y_ref, dv = H.run(d, "Custom waypoints",
                              custom="0:-1.75; 1.0:1.75; 3.0:0.0; 5.0:-1.75",
                              ramp=0.5, T_sim=7.0)
H.print_summary(tr_p, tr_r, dv)
_ = H.plot_static(d, tr_p, tr_r, y_ref, title="custom waypoints, ramp 0.5 s")

In [ ]:
# Batch over plant / noise realisations of the paper manoeuvre:
# does the true error e_k = x_k - z_k ever leave Omega (alpha_k > 1), and how large is it on average?
n_runs = 20
feas = {"HT-MPC": 0, "RT-MPC": 0};  outside = 0;  steps = 0;  alpha_all = []
for seed in range(n_runs):
    tr_p, tr_r, _, _ = H.run(d, "Single lane change (paper)", plant_seed=seed, noise_seed=100 + seed)
    feas["HT-MPC"] += tr_p.ok;  feas["RT-MPC"] += tr_r.ok
    outside += int((tr_p.alpha > 1. + 1e-6).sum());  steps += len(tr_p.alpha)
    alpha_all.append(tr_p.alpha)
alpha_all = np.concatenate(alpha_all)
print(f"feasible runs        : HT-MPC {feas['HT-MPC']}/{n_runs}, RT-MPC {feas['RT-MPC']}/{n_runs}")
print(f"steps with e_k outside Omega : {outside}/{steps}  (bound designed at p = 0.99)")
print(f"alpha_k over all runs: mean {alpha_all.mean():.3f}, 95th pct {np.percentile(alpha_all, 95):.3f}, "
      f"max {alpha_all.max():.3f}")